# Debbi — AICL on a decoder-only transformer (free T4)

End-to-end MVP: train an AICL vocabulary on a code corpus, benchmark it vs BPE, then train ~150M Debbi from scratch. The research question: **does a corpus-learned Unicode-symbol tokenizer beat BPE for code?**

Run Cells → Run all. Free Colab T4 fits comfortably.

In [ ]:
!pip -q install torch datasets sentencepiece
import os, glob
if not os.path.exists('debbie'):
    !git clone --depth 1 https://github.com/vspcoderz/AICL-Debbi.git debbie
%cd debbie
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os, random
random.seed(0)

def fetch_corpus(target='data/corpus_src.txt'):
    if os.path.exists(target) and os.path.getsize(target) > 500_000:
        return target
    collected = []
    # 1) try The Stack python slice (streaming)
    try:
        from datasets import load_dataset
        ds = load_dataset('bigcode/the-stack', data_dir='data/python', split='train', streaming=True)
        it = iter(ds.shuffle(seed=0, buffer_size=1000))
        for _ in range(40):
            sample = next(it)
            collected.append(sample['content'][:8000])
        print('The Stack OK')
    except Exception as e:
        print('The Stack unavailable:', e)
    # 2) fallback: some raw example files from a stable repo
    if len(collected) < 4:
        import urllib.request
        base = 'https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch'
        rels = ['translation/run_translation.py', 'image-classification/run_image_classification.py',
                'question-answering/run_qa.py', 'multiple-choice/run_swag.py',
                'token-classification/run_ner.py', 'text-classification/run_glue.py']
        for r in rels:
            try:
                txt = urllib.request.urlopen(base + '/' + r, timeout=30).read().decode('utf-8')
                collected.append(txt)
            except Exception as e:
                print('fetch failed', r, e)
        print('fallback OK')
    if not collected:
        with open('samples/code_sample.txt') as fh:
            collected = [fh.read()]
    with open(target, 'w', encoding='utf-8') as fh:
        fh.write('\n\n'.join(collected))
    print(f'corpus: {os.path.getsize(target):,} bytes in {len(collected)} files')
    return target

corpus = fetch_corpus()

In [ ]:
# 1) Learn the AICL vocabulary on this corpus
!python tokenizer/train_vocab.py --input {corpus} \
    --output tokenizer/vocabularies/code-vocab.json --size 3000 --min-freq 2

In [ ]:
# 2) Honest benchmark: AICL tokens vs SentencePiece BPE tokens
!python benchmark/vs_bpe.py --corpus {corpus} --size 3000 \
    --vocab-out tokenizer/vocabularies/code-vocab.json

In [ ]:
# 3) Encode the corpus into token ids
!mkdir -p data
!python data/prepare_data.py --input {corpus} \
    --vocab tokenizer/vocabularies/code-vocab.json --out-dir data

In [ ]:
# 4) Sanity: small model, few steps (CPU/GPU-agnostic). Green = harness works.
!python model/train.py --nano --max-steps 40

In [ ]:
# 5) Debbi-150M — the real run (about an hour on a free T4 for ~4k steps)
import os
os.environ['WANDB_DISABLED'] = '1'
# mount Drive to keep checkpoints across sessions
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/debbi-150m'
    os.makedirs(CKPT, exist_ok=True)
except Exception:
    CKPT = 'checkpoints'

!python model/train.py --data-dir data --out-dir {CKPT} --run-name debbi-150m --max-steps 4000

In [ ]:
# 6) Generate a sample
!python model/generate.py --ckpt {CKPT}/debbi-150m/last.pt \
    --vocab tokenizer/vocabularies/code-vocab.json \
    --id-map data/id_map.json \
    --prompt "def quicksort(arr):" --max-new-tokens 120

**Next steps** (see PLAN.md):
1. If AICL beats BPE on the benchmark → scale data + model; if not → iterate the tokenizer first.
2. Resume training across sessions: `python model/train.py --out-dir {CKPT} --run-name debbi-150m` detects `last.pt`.
3. Profile on a held-out split: `python model/train.py` prints eval ppl every `--eval-every` steps.

**Honesty note:** numbers here come from actual runs only. The 40–50% vs-BPE claim is not assumed — the benchmark measures it.